In [2]:
import os
from dotenv import load_dotenv
# Load variables from the .env file
load_dotenv()

# Enable LangChain tracing automatically using the environment
os.environ["LANGCHAIN_TRACING_V2"] = "true"


In [4]:
from langchain_community.document_loaders import WebBaseLoader

web_loader = WebBaseLoader("https://docs.langchain.com/langsmith/home")
data = web_loader.load()
data

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresLLM GatewayPrivate betaOverviewSpend policiesPII and secrets redactionAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUse this file to discover all available pages before exploring fur

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(data)
texts

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresLLM GatewayPrivate betaOverviewSpend policiesPII and secrets redactionAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUse this file to discover all available pages before exploring fur

In [8]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

from langchain_community.vectorstores import FAISS
db = FAISS.from_documents(texts, embeddings)
db

In [10]:
#query from the vector store
query = "LangSmith is a framework-agnostic platform"
docs = db.similarity_search(query)
docs[0].page_content

'all available pages before exploring further.LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with your agent stack.'

In [16]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")

from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    """
    Answer the question based on the following context:
    <context>
    {context}
    </context>
    
    """
    )


In [20]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the question based on the following context:\n    <context>\n    {context}\n    </context>\n    \n    '), additional_kwargs={})])
| ChatOpenAI(output_version=None, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 

In [ ]:
# invoke the chain with the query and the retrieved documents as context (THIS IS WHAT THE RETRIEVER DOES)
from langchain_core.documents import Document
chain.invoke({
    "input": "LangSmith is a framework-agnostic platform",
    "context": [Document(page_content="LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with your agent stack.")]
})


'What is LangSmith?\n\nLangSmith is a platform that is framework-agnostic and provides tools for building, debugging, and deploying AI agents and LLM (Large Language Model) applications. It allows users to trace requests, evaluate outputs, test prompts, and manage deployments, all within one integrated environment with your agent stack.'

In [26]:
retriever = db.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever, chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001BA57AC6380>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the question based on the following context:\n    <context>\n    {context}\n    </context>\n    \n    '), additional_kwargs={})])
      

In [28]:
response = retrieval_chain.invoke({"input": "What is LangSmith?"})
response["answer"]

"What is LangSmith, and what are its main functions? \n\nLangSmith is a framework-agnostic platform designed for building, debugging, and deploying AI agents and LLM (large language model) applications. Its main functions include:\n\n1. **Tracing Requests** - Allowing users to track the processes and operations performed by their AI applications.\n2. **Evaluating Outputs** - Offering tools to assess and ensure the quality and consistency of AI application outputs.\n3. **Testing Prompts** - Facilitating prompt iteration with built-in versioning and collaboration features.\n4. **Managing Deployments** - Providing a platform for deploying agents as Agent Servers, ready to scale in production.\n5. **Observability** - Offering visibility into every step of the application's operations to improve debugging and reliability.\n6. **Integration** - Supporting integrations with various frameworks and providers for enhanced compatibility.\n7. **Security & Compliance** - Ensuring data security and 

In [29]:
response["context"]

[Document(id='00f68f7b-0910-402b-87d0-c5535631b166', metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresLLM GatewayPrivate betaOverviewSpend policiesPII and secrets redactionAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUse this file to discov